In [31]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.model_selection import GridSearchCV
import joblib

# Load Dataset from Excel

In [26]:
print("Loading Dataset...")
df = pd.read_excel("1.xlsx", sheet_name="original")

Loading Dataset...


# Data Preparation - Keep only intelligence scores and profession

In [9]:
print("\nPreparing data...")
# Corrected column names based on the df variable
intelligence_cols = ['Linguistic', 'Musical', 'Bodily', 'Logical - Mathematical',
                    'Spatial-Visualization', 'Interpersonal', 'Intrapersonal', 'Naturalist']
df = df[intelligence_cols + ['Job profession']] # Corrected column name



Preparing data...


# Feature Engineering - Create composite scores

In [10]:
print("\nEngineering features...")
df['Analytical_Score'] = (df['Logical - Mathematical'] + df['Spatial-Visualization']) / 2
df['Creative_Score'] = (df['Musical'] + df['Bodily']) / 2
df['Social_Score'] = (df['Interpersonal'] + df['Intrapersonal']) / 2


Engineering features...


# Split Data

In [11]:
print("\nSplitting data...")
X = df.drop('Job profession', axis=1) # Corrected column name
y = df['Job profession'] # Corrected column name


Splitting data...


In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)


# Encode Labels

In [13]:
print("\nEncoding labels...")
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)


Encoding labels...


# Train Model

In [32]:
print("\nTraining Random Forest with Grid Search...")

# Define the parameter grid
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 3, 5],
    'min_samples_split': [2, 5, 10],
    'class_weight': ['balanced', None]
}

# Create a RandomForestClassifier model
rf = RandomForestClassifier(random_state=42)

# Create GridSearchCV object
grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, cv=5, scoring='accuracy', n_jobs=-1)

# Fit the grid search to the training data
grid_search.fit(X_train, y_train_enc)

# Get the best model and best parameters
model = grid_search.best_estimator_
print("Best parameters found: ", grid_search.best_params_)

print("\n=== Model Training Complete ===")


Training Random Forest with Grid Search...
Best parameters found:  {'class_weight': 'balanced', 'max_depth': None, 'min_samples_split': 2, 'n_estimators': 300}

=== Model Training Complete ===


# Feature Importance

In [34]:
print("\nFeature Importances:")
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': model.feature_importances_
}).sort_values('Importance', ascending=False)
print(feature_importance)


Feature Importances:
                   Feature  Importance
7               Naturalist    0.129208
6            Intrapersonal    0.120322
5            Interpersonal    0.101792
2                   Bodily    0.101780
4    Spatial-Visualization    0.096586
0               Linguistic    0.095326
3   Logical - Mathematical    0.085796
1                  Musical    0.085446
10            Social_Score    0.068528
8         Analytical_Score    0.058870
9           Creative_Score    0.056345


# Evaluate

In [35]:
print("\n=== Evaluation ===")
print("Test Accuracy:", model.score(X_test, y_test_enc))


=== Evaluation ===
Test Accuracy: 0.9819444444444444


# Save Model

In [37]:
print("\nSaving model...")
joblib.dump(model, 'career_model_intelligence_only.pkl')
joblib.dump(le, 'label_encoder_intelligence.pkl')
print("\n=== Model Training Complete ===")
print("Saved files:")
print("- career_model_intelligence_only.pkl")
print("- label_encoder_intelligence.pkl")


Saving model...

=== Model Training Complete ===
Saved files:
- career_model_intelligence_only.pkl
- label_encoder_intelligence.pkl


In [39]:
import joblib
import pandas as pd

print("Loading the saved model and label encoder...")
loaded_model = joblib.load('career_model_intelligence_only.pkl')
loaded_le = joblib.load('label_encoder_intelligence.pkl')
print("Model and label encoder loaded successfully.")

Loading the saved model and label encoder...
Model and label encoder loaded successfully.


In [40]:
print("\nMaking predictions using the loaded model...")

# You can use the same dummy data or provide new data here
dummy_data_loaded = pd.DataFrame({
    'Linguistic': [15, 8, 12],
    'Musical': [10, 18, 5],
    'Bodily': [8, 15, 10],
    'Logical - Mathematical': [18, 7, 14],
    'Spatial-Visualization': [17, 6, 13],
    'Interpersonal': [16, 14, 9],
    'Intrapersonal': [15, 12, 8],
    'Naturalist': [14, 11, 16],
    'Analytical_Score': [(18+17)/2, (7+6)/2, (14+13)/2], # Calculate based on Logical and Spatial
    'Creative_Score': [(10+8)/2, (18+15)/2, (5+10)/2], # Calculate based on Musical and Bodily
    'Social_Score': [(16+15)/2, (14+12)/2, (9+8)/2] # Calculate based on Interpersonal and Intrapersonal
})

# Make predictions using the loaded model
predictions_encoded_loaded = loaded_model.predict(dummy_data_loaded)

# Inverse transform predictions to get job profession names using the loaded label encoder
predictions_loaded = loaded_le.inverse_transform(predictions_encoded_loaded)

print("\nPredictions using the loaded model:")
for i, prediction in enumerate(predictions_loaded):
    print(f"Dummy Data {i+1}: Predicted Job Profession - {prediction}")


Making predictions using the loaded model...

Predictions using the loaded model:
Dummy Data 1: Predicted Job Profession - Interior Decorator
Dummy Data 2: Predicted Job Profession - Dancer
Dummy Data 3: Predicted Job Profession - Archeologist
